In [1]:
!curl -sL -o splits.zip https://github.com/J-SNACKKB/FLIP/raw/main/splits/gb1/splits.zip
!mkdir -p data && unzip -o -q splits.zip -d data && ls data/splits

low_vs_high.csv    one_vs_rest.fasta  three_vs_rest.csv    two_vs_rest.fasta
low_vs_high.fasta  sampled.csv	      three_vs_rest.fasta
one_vs_rest.csv    sampled.fasta      two_vs_rest.csv


In [2]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
SPLIT = "sampled"
AMINO = "ACDEFGHIKLMNPQRSTVWY"
POSITIONS = [38, 39, 40, 53]


class GB1(Dataset):
    def __init__(self, frame):
        idx = [[AMINO.index(s[p]) for p in POSITIONS] for s in frame["sequence"]]
        idx = torch.tensor(idx)                                     # (n, 4) integers
        self.x = nn.functional.one_hot(idx, 20).float().flatten(1)  # (n, 80)
        self.y = torch.tensor(frame["target"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

In [4]:
df = pd.read_csv(f"data/splits/{SPLIT}.csv")
train_ds = GB1(df[df.set == "train"])
test_ds = GB1(df[df.set == "test"])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=512)

x, y = next(iter(train_dl))
print(x.shape, y.shape)

torch.Size([64, 80]) torch.Size([64])


In [5]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(80, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLP().to(device)
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=80, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [6]:
loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)


def evaluate(dl):
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for x, y in dl:
            preds.append(model(x.to(device)).cpu())
            ys.append(y)
    return spearmanr(torch.cat(preds), torch.cat(ys)).statistic

In [7]:
for epoch in range(1, 31):
    model.train()
    total = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(y)
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  train loss {total / len(train_ds):.4f}  test spearman {evaluate(test_dl):.3f}")

epoch  5  train loss 0.1600  test spearman 0.917
epoch 10  train loss 0.0819  test spearman 0.933
epoch 15  train loss 0.0507  test spearman 0.940
epoch 20  train loss 0.0365  test spearman 0.945
epoch 25  train loss 0.0295  test spearman 0.947
epoch 30  train loss 0.0205  test spearman 0.946
